# Data Centre Mapping — Automated Form Submission

This notebook automates the submission of data centre records to the shared Google Form.

---

### Instructions

1. Install dependencies: `pip install requests pandas`
2. Place this notebook in the same folder as your `.xlsx` data file 
3. Set `CSV_FILE` in the **Configuration** cell to the name of your file
4. Run with `SEND = False` first to validate your data — no submissions will be made
5. Fix any errors reported, then re-run with `SEND = True`

---

> ### ⚠️ Critical Warning
> **Do NOT run this notebook more than once on the same dataset with `SEND = True`.**  
> Every execution submits **all valid rows** to the form.  
> Duplicate entries pollute the shared database and must be removed manually by course staff.

---
## Step 1 — Install Dependencies

In [1]:
# Uncomment the line below if dependencies are not yet installed
# !pip install requests pandas

---
## Step 2 — Imports

In [2]:
import re
import time
import requests
import pandas as pd

---
## Step 3 — Configuration

**Only edit this cell.** Set your `.xlsx` filename and toggle `SEND` when ready.

In [9]:
CSV_FILE = "Croatia Cyprus Romania.xlsx"  # Name of your .xlsx data file
SEND     = False                # Set to True ONLY when ready to submit for real
DELAY    = 2.0                  # Seconds between submissions — do not lower this value

---
## Step 4 — Form Endpoints

⚠️ Do not modify this cell.

In [4]:
FORM_ID  = "formID"  # INSERT THE FORM ID HERE
FORM_URL = f"https://docs.google.com/forms/d/e/{FORM_ID}/viewform"
POST_URL = f"https://docs.google.com/forms/d/e/{FORM_ID}/formResponse"

---
## Step 5 — Field Map

Maps Google Form entry IDs to column names in your `.xlsx` file.  
**Do not change the entry IDs on the left.** Adjust the column names on the right only if your `.xlsx` headers differ.

In [5]:
ENTRY_MAP = {
    "entry.320628032":  "Group Number ID",
    "entry.486861776":  "Unique Data Centre ID",
    "entry.1972904144": "Data Centre Name",
    "entry.908908597":  "Operator",
    "entry.673944872":  "Street Address",
    "entry.850496348":  "City",
    "entry.542465020":  "Postal Code / ZIP Code",
    "entry.1365808410": "Country ISO2 code",
    "entry.963378994":  "Coordinates (Latitude, Longitude)",
    "entry.1346551664": "NUTS3 Region Code",
    "entry.1125580182": "Primary Source Confirming the Data Centre",
    "entry.1771583486": "News Articles Mentioning the Data Centre",
    "entry.1614300136": "Individuals Mentioned in News",
    "entry.1842784360": "Operating Company (Legal Entity / Permit Applicant)",
    "entry.410056195":  "Parent Company",
    "entry.2061530222": "Is the operator or its parent company backed or owned by a private equity fund or asset manager?",
    "entry.1341620919": "Source for Ownership Information (URL)",
    "entry.360157987":  "Construction Year",
    "entry.1476933402": "Estimated IT Load (MW)",
    "entry.1163342674": "Generator Type",
    "entry.399398013":  "Total Generator Rated Capacity (MW)",
    "entry.985202037":  "Annual Water Consumption",
    "entry.425180581":  "Water Source (Basin / Aquifer)",
    "entry.409780800":  "CO2e Emissions (tons per year)",
    "entry.539905950":  "Information Source(s)",
}

---
## Step 6 — Validation Rules

⚠️ Do not modify this cell.  
These checks replicate **exactly** the constraints defined in the Google Form. Removing or weakening them will allow invalid data into the shared database.

In [6]:
# Regex patterns copied verbatim from the form HTML source
RE_ISO2 = re.compile(
    r"^(BE|BG|CZ|DK|DE|EE|IE|EL|ES|FR|HR|IT|CY|LV|LT|LU|HU|MT|NL|AT|PL|PT"
    r"|RO|SI|SK|FI|SE|IS|LI|NO|CH|ME|MK|AL|RS|TR|UA|XK)$"
)
RE_COORDS = re.compile(
    r"^-?([1-8]?\d(\.\d+)?|90(\.0+)?),\s-?((1[0-7]\d)|([1-9]?\d)|180)(\.\d+)?$"
)

# Allowed values for radio button fields
VALID_PE_VALUES = {"Yes", "No", "Unknown"}
VALID_GEN_VALUES = {
    "Diesel",
    "Natural Gas",
    "HVO (Hydrotreated Vegetable Oil)",
    "Biogas",
    "Dual-fuel (Diesel + Gas)",
    "Battery-only backup",
    "Hydrogen",
    "Unknown",
}

# Mandatory fields
REQUIRED_FIELDS = [
    "Group Number ID",
    "Unique Data Centre ID",
    "Data Centre Name",
    "Operator",
    "Street Address",
    "City",
    "Postal Code / ZIP Code",
    "Country ISO2 code",
    "Coordinates (Latitude, Longitude)",
    "NUTS3 Region Code",
    "Primary Source Confirming the Data Centre",
    "News Articles Mentioning the Data Centre",
    "Individuals Mentioned in News",
]


def validate_row(row):
    """
    Replicates all validation rules from the Google Form.
    Returns a list of error messages. An empty list means the row is valid.
    """
    errors = []

    # 1. Mandatory fields must not be empty
    for field in REQUIRED_FIELDS:
        if str(row.get(field, "")).strip() == "":
            errors.append(f"[REQUIRED] Empty mandatory field: '{field}'")

    # 2. Group Number ID — max 10 characters
    group_id = str(row.get("Group Number ID", "")).strip()
    if len(group_id) > 10:
        errors.append(
            f"[FORMAT] 'Group Number ID' is too long "
            f"({len(group_id)} chars, max 10): '{group_id}'"
        )

    # 3. Country ISO2 code — must match allowed values
    iso2 = str(row.get("Country ISO2 code", "")).strip()
    if iso2 and not RE_ISO2.match(iso2):
        errors.append(
            f"[FORMAT] Invalid 'Country ISO2 code': '{iso2}'\n"
            f"         Allowed: BE BG CZ DK DE EE IE EL ES FR HR IT CY LV LT LU\n"
            f"         HU MT NL AT PL PT RO SI SK FI SE IS LI NO CH ME MK AL RS TR UA XK"
        )

    # 4. Coordinates — must match lat/lon format
    coords = str(row.get("Coordinates (Latitude, Longitude)", "")).strip()
    if coords and not RE_COORDS.match(coords):
        errors.append(
            f"[FORMAT] Invalid 'Coordinates': '{coords}'\n"
            f"         Use Google Maps format: 45.4642, 9.1900  (lat, space, lon)"
        )

    # 5. Private equity question — only allowed radio values
    pe_col = (
        "Is the operator or its parent company backed or owned by a "
        "private equity fund or asset manager?"
    )
    pe = str(row.get(pe_col, "")).strip()
    if pe and pe not in VALID_PE_VALUES:
        errors.append(
            f"[FORMAT] Invalid PE answer: '{pe}'\n"
            f"         Allowed values: Yes / No / Unknown"
        )

    # 6. Generator Type — only allowed radio values
    gen = str(row.get("Generator Type", "")).strip()
    if gen and gen not in VALID_GEN_VALUES:
        errors.append(
            f"[FORMAT] Invalid 'Generator Type': '{gen}'\n"
            f"         Allowed: {' / '.join(sorted(VALID_GEN_VALUES))}"
        )

    return errors

---
## Step 7 — Session Token Helper

⚠️ Do not modify this cell.  
Fetches the `fbzx` session token dynamically from the live form page. This value is regenerated by Google on every page load — using a hardcoded value will cause submissions to fail.

In [7]:
def get_fbzx():
    try:
        r = requests.get(FORM_URL, timeout=10)
        match = re.search(r'name="fbzx" value="(-?\d+)"', r.text)
        if match:
            return match.group(1)
        match = re.search(r'data-shuffle-seed="(-?\d+)"', r.text)
        if match:
            return match.group(1)
    except Exception as e:
        print(f"  WARNING: could not fetch fbzx dynamically ({e}). Using fallback.")
    return "-9151365495411357258"  # last known value from form source

---
## Step 8 — Load Data

In [11]:
df = pd.read_excel(CSV_FILE).fillna("")
# If you have more than one spreedsheet
# df = pd.read_excel(CSV_FILE, sheet_name="SheetName").fillna("")

# If you have a csv (coma separated value)
# df = pd.read_csv(CSV_FILE, sep=",").fillna("")

# Rename email column if exported from a Google Form in Italian
if "Indirizzo email" in df.columns and "Email" not in df.columns:
    df.rename(columns={"Indirizzo email": "Email"}, inplace=True)

print(f"Columns found in the .tsv:\n  {list(df.columns)}\n")
print(f"Total rows to process: {len(df)}")
print("=" * 65)

Columns found in the .tsv:
  ['country', 'market', 'facility name', 'Informazioni cronologiche', 'Group Number ID', 'Unique Data Centre ID', 'Data Centre Name', 'Operator', 'Street Address', 'locality', 'City', 'Postal Code / ZIP Code', 'Country ISO2 code ', 'Coordinates (Latitude, Longitude)', 'NUTS3 Region Code', 'Primary Source Confirming the Data Centre', 'News Articles Mentioning the Data Centre', 'Individuals Mentioned in News', 'Operating Company (Legal Entity / Permit Applicant)', 'Parent Company', 'Is the operator or its parent company backed or owned by a private equity fund or asset manager?', 'Source for Ownership Information (URL)', 'Construction Year', 'est_year', 'Estimated IT Load (MW)', 'power_mw', 'whitespace_sqm', 'carrier_neutral', 'Generator Type ', 'Total Generator Rated Capacity (MW)', 'Annual Water Consumption', 'Water Source (Basin / Aquifer)', 'CO2e Emissions (tons per year)', 'Information Source(s)', 'Email']

Total rows to process: 100


In [21]:
# Verify every column is in the dictionary ENTRY_MAP at step 5 
for col in df.columns:
    if not col in ENTRY_MAP.values():
        print("'"+col+"'")

'country'
'market'
'facility name'
'Informazioni cronologiche'
'locality'
'Country ISO2 code '
'est_year'
'power_mw'
'whitespace_sqm'
'carrier_neutral'
'Generator Type '
'Email'


---
## Step 9 — Validate & Submit

- With `SEND = False`: validates all rows and prints a full error report. Nothing is submitted.
- With `SEND = True`: submits all valid rows to the Google Form.

> ⚠️ Run with `SEND = True` **exactly once** per dataset.

In [12]:
ok_count     = 0
failed_valid = []   # rows skipped due to validation errors
failed_send  = []   # rows where the HTTP request failed

# Fetch the fbzx token once before the loop
fbzx = get_fbzx()
print(f"fbzx session token: {fbzx}\n")

for i, row in df.iterrows():
    print(f"\nRow {i+1}/{len(df)} — {str(row.get('Data Centre Name', '???'))[:55]}")

    # --- Validate ---
    errors = validate_row(row)
    if errors:
        print(f"  x SKIPPED — {len(errors)} validation error(s):")
        for e in errors:
            for line in e.split("\n"):
                print(f"    {line}")
        failed_valid.append(i + 1)
        continue

    print("  v Validation passed")

    # --- Build POST payload ---
    payload = {
        "emailAddress":        str(row.get("Email", "")).strip(),
        "fvv":                 "1",
        "pageHistory":         "0,1,2,3",
        "fbzx":                fbzx,
        "submissionTimestamp": "-1",
    }
    for entry_id, col_name in ENTRY_MAP.items():
        if col_name in df.columns:
            val = str(row[col_name]).strip()
            if val:
                payload[entry_id] = val

    # --- Submit or dry-run ---
    if SEND:
        try:
            r = requests.post(
                POST_URL, data=payload, allow_redirects=False, timeout=15
            )
            if r.status_code in (200, 302):
                print("  -> Submitted successfully")
                ok_count += 1
            else:
                print(f"  -> WARNING: unexpected HTTP status {r.status_code}")
                failed_send.append(i + 1)
        except Exception as e:
            print(f"  -> CONNECTION ERROR: {e}")
            failed_send.append(i + 1)
        time.sleep(DELAY)
    else:
        fields = [k for k in payload if k.startswith("entry")]
        print(f"  -> TEST MODE — {len(fields)} fields ready to send")

fbzx session token: -9151365495411357258


Row 1/100 — HR-ZOO OS
  x SKIPPED — 4 validation error(s):
    [REQUIRED] Empty mandatory field: 'Country ISO2 code'
    [REQUIRED] Empty mandatory field: 'Coordinates (Latitude, Longitude)'
    [REQUIRED] Empty mandatory field: 'News Articles Mentioning the Data Centre'
    [REQUIRED] Empty mandatory field: 'Individuals Mentioned in News'

Row 2/100 — HR-ZOO RI
  x SKIPPED — 4 validation error(s):
    [REQUIRED] Empty mandatory field: 'Country ISO2 code'
    [REQUIRED] Empty mandatory field: 'Coordinates (Latitude, Longitude)'
    [REQUIRED] Empty mandatory field: 'News Articles Mentioning the Data Centre'
    [REQUIRED] Empty mandatory field: 'Individuals Mentioned in News'

Row 3/100 — HR-ZOO ST
  x SKIPPED — 4 validation error(s):
    [REQUIRED] Empty mandatory field: 'Country ISO2 code'
    [REQUIRED] Empty mandatory field: 'Coordinates (Latitude, Longitude)'
    [REQUIRED] Empty mandatory field: 'News Articles Mentioning the Data Centre'

---
## Step 10 — Summary

In [13]:
print("\n" + "=" * 65)
print("FINAL SUMMARY")
print(f"  Total rows in file:          {len(df)}")
print(f"  Skipped (validation errors): {len(failed_valid)}")
print(f"  Failed (HTTP errors):        {len(failed_send)}")
if SEND:
    print(f"  Successfully submitted:      {ok_count}")
else:
    print("  (SEND = False — nothing was actually submitted)")

if failed_valid:
    print(f"\n  Rows with validation errors: {failed_valid}")
if failed_send:
    print(f"  Rows with HTTP errors:       {failed_send}")


FINAL SUMMARY
  Total rows in file:          100
  Skipped (validation errors): 100
  Failed (HTTP errors):        0
  (SEND = False — nothing was actually submitted)

  Rows with validation errors: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]


In [ ]:
'Country ISO2 code '

Index(['country', 'market', 'facility name', 'Informazioni cronologiche',
       'Group Number ID', 'Unique Data Centre ID', 'Data Centre Name',
       'Operator', 'Street Address', 'locality', 'City',
       'Postal Code / ZIP Code', 'Country ISO2 code ',
       'Coordinates (Latitude, Longitude)', 'NUTS3 Region Code',
       'Primary Source Confirming the Data Centre',
       'News Articles Mentioning the Data Centre',
       'Individuals Mentioned in News',
       'Operating Company (Legal Entity / Permit Applicant)', 'Parent Company',
       'Is the operator or its parent company backed or owned by a private equity fund or asset manager?',
       'Source for Ownership Information (URL)', 'Construction Year',
       'est_year', 'Estimated IT Load (MW)', 'power_mw', 'whitespace_sqm',
       'carrier_neutral', 'Generator Type ',
       'Total Generator Rated Capacity (MW)', 'Annual Water Consumption',
       'Water Source (Basin / Aquifer)', 'CO2e Emissions (tons per year)',
       'I